# CFFair — Batch Pipelines (deployable, abduction-based)

Implements Kusner et al. (2017)'s Level 3 method faithfully: an **abduction** step infers
the part of each observable feature that S cannot explain (the residual/error term), fitted
only on training data. The classifier then predicts Y from those residuals alone — never
from raw S or raw X directly, and never from ground-truth U.

**Test-time inputs: S0 + X only** — same as FairPFN, XGBoost, CLAIRE, and SRCVAE, so this
model is directly comparable in the benchmark ranking. (An Oracle variant that uses
ground-truth U as a theoretical ceiling was considered and intentionally dropped — this
notebook implements only the deployable model.)

Each pipeline has a **TRIAL** cell (1-2 files, writes to a separate `_TRIAL.csv`) before
the full batch run — delete the TRIAL cells once confirmed working.

**Paths:**
- Synthetic data: `Data_generation/SF_and_Hidden_Nodes_Data_Simulation/results/generated_160_csv_10_seeds/full_data`
- Semi-synthetic data: `Data_generation/HR_Simulation_Datasets`
- Results output: `model_results/CFFair_syn_results.csv` and `model_results/CFFair_semi_syn_results.csv`


In [ ]:
# ==========================================
# LIBRARIES
# ==========================================
import os
import glob
import re
import warnings

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings('ignore')

# Shared paths (relative to this notebook's location: Fairness_models/)
SYN_INPUT_FOLDER = "../Data_generation/SF_and_Hidden_Nodes_Data_Simulation/results/generated_160_csv_10_seeds/full_data"
SEMI_INPUT_FOLDER = "../Data_generation/HR_Simulation_Datasets"
OUTPUT_DIR = "../model_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## CFFair Architecture (Abduction + Classifier)

In [ ]:
# ==========================================
# STEP 1: CFFAIR CLASSIFIER (operates on ABDUCTED RESIDUALS, never raw S or X)
# ==========================================
class CFFair_Classifier(nn.Module):
    """
    Predicts Y from the abducted residuals (the part of X that S cannot explain).
    Input dimension matches the number of X columns, since one residual is computed
    per observable feature.
    """
    def __init__(self, residual_dim, hidden_dim=64):
        super(CFFair_Classifier, self).__init__()
        self.fc1 = nn.Linear(residual_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, residual):
        h = F.relu(self.fc1(residual))
        h = F.relu(self.fc2(h))
        return torch.sigmoid(self.out(h))


def abduct_residuals(S_train, X_train, S_test, X_test):
    """
    Abduction step (Kusner et al., 2017, Level 3): fits X ~ S using linear regression
    on TRAINING data only, then computes residuals for both train and test as
    X - predicted(X | S). These residuals approximate the error term U_i in the
    paper's additive-noise causal model — the part of each feature not explained by S.

    Fitting only on training data (rather than train+test combined, as the original
    paper's reference implementation does) keeps this consistent with a strict
    train/test split and avoids any test-set leakage into the abduction model itself.
    """
    abduction_model = LinearRegression()
    abduction_model.fit(S_train, X_train)

    residual_train = X_train - abduction_model.predict(S_train)
    residual_test = X_test - abduction_model.predict(S_test)

    return residual_train, residual_test


def train_cffair(model, residual_tensor, Y_tensor, epochs=100, batch_size=128, lr=1e-3):
    """
    Trains the classifier to predict Y from the abducted residuals only.
    """
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = TensorDataset(residual_tensor, Y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        for batch_r, batch_y in dataloader:
            optimizer.zero_grad()
            y_pred = model(batch_r)
            loss = F.binary_cross_entropy(y_pred, batch_y)
            loss.backward()
            optimizer.step()

    return model


def predict_cffair(model, residual_tensor):
    """
    Generates predictions from abducted residuals. At no point does this touch
    raw S, raw X, or ground-truth U -- only the residual computed at abduction time.
    """
    model.eval()
    with torch.no_grad():
        probs = model(residual_tensor)
        preds = (probs >= 0.5).float()
        return probs.numpy().flatten(), preds.numpy().flatten()


## Purely Synthetic Data

### 🧪 TRIAL — run on 1-2 files only (delete this cell once confirmed working)

In [ ]:
# --- TRIAL: synthetic, 2 files only, writes to a separate _TRIAL.csv ---
N_TRIAL_FILES = 2
trial_output_file = os.path.join(OUTPUT_DIR, "CFFair_syn_results_TRIAL.csv")

dataset_files_trial = glob.glob(os.path.join(SYN_INPUT_FOLDER, "*.csv"))[:N_TRIAL_FILES]
print(f"[TRIAL] Found {len(dataset_files_trial)} file(s) to test with.")

trial_results = []

for file_path in dataset_files_trial:
    dataset_name = os.path.basename(file_path)
    print(f"\n[TRIAL] Processing: {dataset_name} with CFFair...")

    try:
        df = pd.read_csv(file_path)

        # --- DYNAMIC FEATURE DISCOVERY ---
        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith(('U', 'H', 'C'))]  # logged only, never used

        if 'Y' not in df.columns or len(s_cols) == 0 or len(x_cols) == 0:
            print("  [SKIPPED] Missing 'Y', 'S', or 'X' columns.")
            continue

        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        S_data = df[[s_target]].values
        X_features = df[x_cols].values
        y_data = df['Y'].values.reshape(-1, 1)

        train_size = min(1000, int(len(X_features) * 0.8))

        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        # --- ABDUCTION: infer residuals (fit on train only) ---
        residual_train, residual_test = abduct_residuals(S_train, X_train, S_test, X_test)

        residual_train_t = torch.tensor(residual_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        residual_test_t = torch.tensor(residual_test, dtype=torch.float32)

        residual_dim = residual_train.shape[1]

        # --- TRAIN & PREDICT ---
        classifier = CFFair_Classifier(residual_dim=residual_dim)
        classifier = train_cffair(classifier, residual_train_t, y_train_t, epochs=100)

        prob_preds, predictions = predict_cffair(classifier, residual_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        # --- METRICS ---
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        trial_results.append({
            "model_name": "CFFair", "name_dataset": dataset_name,
            "n_S": len(s_cols), "n_X": len(x_cols), "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4), "Accuracy": round(acc, 4),
            "Precision": round(prec, 4), "Recall": round(rec, 4), "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4), "Pos_Rate_S0": round(rate_0, 4)
        })
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

trial_df = pd.DataFrame(trial_results)
trial_df.to_csv(trial_output_file, index=False)
print(f"\n[TRIAL] Done. Results written to {trial_output_file}")
trial_df


### Full batch pipeline — Purely Synthetic Data

In [ ]:
# ==========================================
# SYNTHETIC PIPELINE INTEGRATION
# ==========================================
input_folder = SYN_INPUT_FOLDER
output_file = os.path.join(OUTPUT_DIR, "CFFair_syn_results.csv")

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with CFFair...")

    try:
        df = pd.read_csv(file_path)

        # --- DYNAMIC FEATURE DISCOVERY ---
        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith(('U', 'H', 'C'))]  # logged only, never used

        if 'Y' not in df.columns or len(s_cols) == 0 or len(x_cols) == 0:
            print(f"  [SKIPPED] Missing 'Y', 'S', or 'X' columns.")
            continue

        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        S_data = df[[s_target]].values
        X_features = df[x_cols].values
        y_data = df['Y'].values.reshape(-1, 1)

        train_size = min(1000, int(len(X_features) * 0.8))

        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        # --- ABDUCTION: infer residuals (fit on train only, applied to test) ---
        residual_train, residual_test = abduct_residuals(S_train, X_train, S_test, X_test)

        residual_train_t = torch.tensor(residual_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        residual_test_t = torch.tensor(residual_test, dtype=torch.float32)

        residual_dim = residual_train.shape[1]

        # --- TRAIN & PREDICT (classifier never sees raw S, raw X, or ground-truth U) ---
        classifier = CFFair_Classifier(residual_dim=residual_dim)
        classifier = train_cffair(classifier, residual_train_t, y_train_t, epochs=100)

        prob_preds, predictions = predict_cffair(classifier, residual_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        # --- 1. PREDICTION METRICS ---
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        # --- 2. FAIRNESS METRICS ---
        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)

        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0

        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        # --- RECORD DATA ---
        master_results.append({
            "model_name": "CFFair",
            "name_dataset": dataset_name,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })

        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

# ==========================================
# SAVE AGGREGATED RESULTS
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")


## Semi-Synthetic (HR) Data

### 🧪 TRIAL — run on 1-2 files only (delete this cell once confirmed working)

In [ ]:
# --- TRIAL: semi-synthetic, 2 files only, writes to a separate _TRIAL.csv ---
N_TRIAL_FILES = 2
trial_output_file_semi = os.path.join(OUTPUT_DIR, "CFFair_semi_syn_results_TRIAL.csv")

dataset_files_trial_semi = glob.glob(os.path.join(SEMI_INPUT_FOLDER, "*.csv"))[:N_TRIAL_FILES]
print(f"[TRIAL] Found {len(dataset_files_trial_semi)} file(s) to test with.")

trial_results_semi = []

for file_path in dataset_files_trial_semi:
    dataset_name = os.path.basename(file_path)
    print(f"\n[TRIAL] Processing: {dataset_name} with CFFair...")

    try:
        df = pd.read_csv(file_path)

        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue
        y_data = df[y_cols[0]].values.reshape(-1, 1)

        # CFFair requires S to perform abduction -- skip if absent
        s_cols = [col for col in df.columns if col.startswith('S') or col.startswith('G')]
        if not s_cols:
            print("  [SKIPPED] No 'S' or 'G' column found. CFFair requires a protected attribute for abduction.")
            continue
        s_target = s_cols[0]
        S_data = df[[s_target]].values

        x_cols = [col for col in df.columns if col.startswith('X_')]
        if not x_cols:
            print("  [SKIPPED] No 'X_' columns found.")
            continue
        X_features = df[x_cols].values

        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]  # logged only, never used

        train_size = min(1000, int(len(X_features) * 0.8))

        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        residual_train, residual_test = abduct_residuals(S_train, X_train, S_test, X_test)

        residual_train_t = torch.tensor(residual_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        residual_test_t = torch.tensor(residual_test, dtype=torch.float32)

        residual_dim = residual_train.shape[1]

        classifier = CFFair_Classifier(residual_dim=residual_dim)
        classifier = train_cffair(classifier, residual_train_t, y_train_t, epochs=100)

        prob_preds, predictions = predict_cffair(classifier, residual_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        trial_results_semi.append({
            "model_name": "CFFair", "name_dataset": dataset_name,
            "bias_level": bias_level, "threshold": threshold, "data_type": data_type,
            "n_S": len(s_cols), "n_X": len(x_cols), "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4), "Accuracy": round(acc, 4),
            "Precision": round(prec, 4), "Recall": round(rec, 4), "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4), "Pos_Rate_S0": round(rate_0, 4)
        })
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

trial_df_semi = pd.DataFrame(trial_results_semi)
trial_df_semi.to_csv(trial_output_file_semi, index=False)
print(f"\n[TRIAL] Done. Results written to {trial_output_file_semi}")
trial_df_semi


### Full batch pipeline — Semi-Synthetic (HR) Data

In [ ]:
# ==========================================
# SEMI-SYNTHETIC PIPELINE INTEGRATION
# ==========================================
input_folder = SEMI_INPUT_FOLDER
output_file = os.path.join(OUTPUT_DIR, "CFFair_semi_syn_results.csv")

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} semi-synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with CFFair...")

    try:
        df = pd.read_csv(file_path)

        # --- FILENAME PARAMETER EXTRACTION ---
        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        # --- DYNAMIC FEATURE DISCOVERY ---
        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue
        y_data = df[y_cols[0]].values.reshape(-1, 1)

        # CFFair requires S to perform abduction -- skip if absent
        s_cols = [col for col in df.columns if col.startswith('S') or col.startswith('G')]
        if not s_cols:
            print("  [SKIPPED] No 'S' or 'G' column found. CFFair requires a protected attribute for abduction.")
            continue
        s_target = s_cols[0]
        S_data = df[[s_target]].values

        x_cols = [col for col in df.columns if col.startswith('X_')]
        if not x_cols:
            print("  [SKIPPED] No 'X_' columns found.")
            continue
        X_features = df[x_cols].values

        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]  # logged only, never used

        train_size = min(1000, int(len(X_features) * 0.8))

        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        # --- ABDUCTION: infer residuals (fit on train only, applied to test) ---
        residual_train, residual_test = abduct_residuals(S_train, X_train, S_test, X_test)

        residual_train_t = torch.tensor(residual_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        residual_test_t = torch.tensor(residual_test, dtype=torch.float32)

        residual_dim = residual_train.shape[1]

        # --- TRAIN & PREDICT ---
        classifier = CFFair_Classifier(residual_dim=residual_dim)
        classifier = train_cffair(classifier, residual_train_t, y_train_t, epochs=100)

        prob_preds, predictions = predict_cffair(classifier, residual_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        # --- 1. PREDICTION METRICS ---
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        # --- 2. FAIRNESS METRICS ---
        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)

        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0

        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        # --- RECORD DATA ---
        master_results.append({
            "model_name": "CFFair",
            "name_dataset": dataset_name,
            "bias_level": bias_level,
            "threshold": threshold,
            "data_type": data_type,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })

        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

# ==========================================
# SAVE AGGREGATED RESULTS
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")
